# Brooklyn ambulance travel times

Build a directed driving-distance matrix from the candidate sites in your input CSV to the ZIP representative points. Valhalla uses OpenStreetMap roads with its standard car access, one-way and turn rules. We request distance-based costing and disable hierarchy pruning. The route is between road positions selected by the router. The straight-line offsets to those positions are recorded separately.

The model uses **15 mph** for every route. We discard the router's travel-time estimate and calculate `minutes = 60 * network_miles / 15`. This is a driving-time proxy for a hypothetical standby-site study.

Put this notebook in the repository root and run it with a Python kernel. It uses the Python standard library and requires internet access. The default inputs are `data/candidate_sites.csv` and `data/demand.csv`. To route a different shortlist, replace the candidate CSV or change `CANDIDATE_FILE` in the first code cell, then run all cells. The number of candidates comes from the CSV.

The routing notebook requires a unique, nonempty `site_id` per row and decimal-degree `lat` and `lon` columns. It also accepts the repo's `name`, `address`, and `snap_miles` columns. Keep a `name` column for the Julia notebook's results display. Example header and row, using illustrative coordinates:

```csv
site_id,name,address,lat,lon
NEW_001,Example candidate,,40.695450,-73.982843
```

The notebook writes `data/travel_times_routed.csv` after validating all pairs. In `Modelling.ipynb`, point `CANDIDATE_FILE` at the same candidate CSV and `TRAVEL_TIME_FILE` at the newly generated matrix.

Routing by [Valhalla](https://valhalla.github.io/valhalla/) via the FOSSGIS demo service. Map data © [OpenStreetMap contributors](https://www.openstreetmap.org/copyright). [Fix the map](https://www.openstreetmap.org/fixthemap).

In [1]:
import csv
import json
import math
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

DATA = Path("data")
CANDIDATE_FILE = DATA / "candidate_sites.csv"
DEMAND_FILE = DATA / "demand.csv"
OUTPUT = DATA / "travel_times_routed.csv"
MATRIX_URL = "https://valhalla1.openstreetmap.de/sources_to_targets"
SPEED_MPH = 15.0
METERS_PER_MILE = 1609.344
MAX_SNAP_METERS = 250
SITES_PER_REQUEST = 2  # Public service limit: at most 100 pairs per request.

def read_points(path, id_column):
    points = []
    seen_ids = set()
    with path.open(encoding="utf-8-sig", newline="") as source:
        reader = csv.DictReader(source)
        missing = {id_column, "lat", "lon"} - set(reader.fieldnames or ())
        if missing:
            raise ValueError(f"{path} is missing columns: {', '.join(sorted(missing))}")
        for row in reader:
            point_id = (row[id_column] or "").strip()
            if not point_id or point_id in seen_ids:
                raise ValueError(f"Missing or duplicate {id_column} on row {reader.line_num} of {path}")
            try:
                lat, lon = float(row["lat"]), float(row["lon"])
            except (TypeError, ValueError) as error:
                raise ValueError(f"Invalid coordinates for {point_id} in {path}") from error
            if not (-90 <= lat <= 90 and -180 <= lon <= 180):
                raise ValueError(f"Invalid coordinates for {point_id} in {path}")
            points.append({"id": point_id, "lat": lat, "lon": lon})
            seen_ids.add(point_id)
    if not points:
        raise ValueError(f"{path} has no input rows")
    return points

sites = read_points(CANDIDATE_FILE, "site_id")
areas = read_points(DEMAND_FILE, "area_id")
print(f"Candidates: {CANDIDATE_FILE}")
print(f"{len(sites)} sites × {len(areas)} demand areas = {len(sites) * len(areas):,} routes")

Candidates: data/candidate_sites.csv
20 sites × 37 demand areas = 740 routes


In [2]:
def straight_line_meters(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(math.radians, (lat1, lon1, lat2, lon2))
    a = math.sin((lat2 - lat1) / 2)**2
    a += math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2)**2
    return 2 * 6_371_008.8 * math.asin(math.sqrt(min(1.0, a)))

def matrix_location(point):
    return {"lat": point["lat"], "lon": point["lon"], "search_cutoff": MAX_SNAP_METERS}

def request_matrix(batch, areas):
    payload = {
        "sources": [matrix_location(site) for site in batch],
        "targets": [matrix_location(area) for area in areas],
        "costing": "auto",
        "costing_options": {"auto": {
            "shortest": True,
            "disable_hierarchy_pruning": True,
        }},
        "units": "kilometers",
        "verbose": True,
    }
    request = Request(
        MATRIX_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "User-Agent": "15C57-Brooklyn-EMS-Study/1.0",
            "X-Client-Id": "15C57-Brooklyn-EMS-Study",
        },
    )
    try:
        with urlopen(request, timeout=60) as response:
            result = json.load(response)
    except HTTPError as error:
        detail = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Routing request failed ({error.code}): {detail}") from error
    except (URLError, TimeoutError) as error:
        raise RuntimeError(f"Could not reach the routing service: {error}") from error

    if result.get("warnings"):
        print("Router warnings:", result["warnings"])
    matrix = result.get("sources_to_targets")
    if result.get("units") != "kilometers" or not isinstance(matrix, list):
        raise ValueError("Expected a verbose distance matrix in kilometers.")
    if len(matrix) != len(batch) or any(len(row) != len(areas) for row in matrix):
        raise ValueError("Router returned an incomplete matrix.")
    return matrix

def route_record(site, area, route):
    pair = f"{site['id']} → {area['id']}"
    distance_km = route.get("distance")
    cost_meters = route.get("cost")
    if distance_km is None or not math.isfinite(distance_km) or distance_km < 0:
        raise ValueError(f"No valid driving distance for {pair}")
    # Distance-based auto costing should report cost in meters, within API rounding.
    if cost_meters is None or not math.isfinite(cost_meters) or abs(cost_meters - 1_000 * distance_km) > 1.1:
        raise ValueError(f"Router cost does not match distance-based costing for {pair}")

    site_snap = straight_line_meters(site["lat"], site["lon"], route["begin_lat"], route["begin_lon"])
    area_snap = straight_line_meters(area["lat"], area["lon"], route["end_lat"], route["end_lon"])
    if max(site_snap, area_snap) > MAX_SNAP_METERS + 1:
        raise ValueError(f"Road snap exceeds {MAX_SNAP_METERS} m for {pair}")

    route_miles = 1_000 * distance_km / METERS_PER_MILE
    return {
        "site_id": site["id"],
        "area_id": area["id"],
        "euclid_miles": straight_line_meters(site["lat"], site["lon"], area["lat"], area["lon"]) / METERS_PER_MILE,
        "network_miles": route_miles,
        "minutes": 60 * route_miles / SPEED_MPH,
        "site_snap_meters": site_snap,
        "area_snap_meters": area_snap,
    }

In [3]:
records = []
for offset in range(0, len(sites), SITES_PER_REQUEST):
    if offset:
        time.sleep(1.1)
    batch = sites[offset:offset + SITES_PER_REQUEST]
    matrix = request_matrix(batch, areas)
    for source_index, row in enumerate(matrix):
        for target_index, route in enumerate(row):
            if (route.get("from_index"), route.get("to_index")) != (source_index, target_index):
                raise ValueError("Router returned a route in an unexpected matrix position.")
            records.append(route_record(batch[source_index], areas[target_index], route))
    print(f"Routed {offset + len(batch)} of {len(sites)} sites")

expected_pairs = {(site["id"], area["id"]) for site in sites for area in areas}
actual_pairs = {(row["site_id"], row["area_id"]) for row in records}
if len(records) != len(expected_pairs) or actual_pairs != expected_pairs:
    raise ValueError("Expected exactly one route per candidate site and demand area.")

temporary_output = OUTPUT.with_suffix(".tmp")
with temporary_output.open("w", encoding="utf-8", newline="") as destination:
    writer = csv.DictWriter(destination, fieldnames=list(records[0]))
    writer.writeheader()
    writer.writerows(records)
temporary_output.replace(OUTPUT)

print(f"Wrote {len(records):,} routes to {OUTPUT}")
print(f"Generated at {datetime.now(timezone.utc).isoformat(timespec='seconds')}")
print(f"Modeled drive times: {min(r['minutes'] for r in records):.2f}–{max(r['minutes'] for r in records):.2f} min")
print(f"Longest road snap: {max(max(r['site_snap_meters'], r['area_snap_meters']) for r in records):.1f} m")

Routed 2 of 20 sites
Routed 4 of 20 sites
Routed 6 of 20 sites
Routed 8 of 20 sites
Routed 10 of 20 sites
Routed 12 of 20 sites
Routed 14 of 20 sites
Routed 16 of 20 sites
Routed 18 of 20 sites
Routed 20 of 20 sites
Wrote 740 routes to data/travel_times_routed.csv
Generated at 2026-11-20T16:46:48+00:00
Modeled drive times: 1.46–51.98 min
Longest road snap: 73.9 m
